### Create NWO work-funder/award linkages (funder-reported)

Resolves the per-project `products` that NWO researchers report (captured as
`products_json` on `openalex.awards.nwo_projects_raw`) to OpenAlex works, and writes a
junction table `openalex.awards.nwo_work_funders` analogous to `crossref_work_funders`:
one row per `(work_id, funder_id)` with the NWO `award_ids` (= project ids).

**Work resolution (two precision-ordered paths — see oxjobs #244 EXPLORE):**
- **Path 1 (DOI):** `products[].url_open_access` carries no clean DOI field; salvage one from
  the (whitespace-corrupted, leading-junk) URL, then join `openalex.works.openalex_works.doi`.
  ~96% of recovered DOIs match.
- **Path 2 (URL -> location):** non-DOI URLs are matched against `openalex.works.locations.urls`;
  accepted only when the URL maps to **exactly one** work (`best_doi`). 97% precision, low recall.

**Award entities are NOT created here** — NWO awards already exist in `openalex_awards`
(provenance `nwopen`, via the `NWO_Awards` task). We only confirm the award exists and emit
the edge. The downstream `WorkAwards` notebook joins this junction to the existing entity.

Patents (`Octrooi`) and contracts (`Contract`) are excluded; other non-work types simply
fail to resolve. Citation fallback (Path 3) and a data re-ingest are tracked in #244 as
follow-on levers if coverage (~74% of projects on the current stale 14.6K capture) must rise.

Feeds: **WorkAwards** (new `nwo_work_funder_awards` leg, joins to `openalex_awards`).

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nwo_work_funders
USING delta
AS
WITH products AS (
  SELECT
    pr.project_id,
    p.prod['url_open_access'] AS url_oa,
    p.prod['type']            AS product_type
  FROM openalex.awards.nwo_projects_raw pr
  LATERAL VIEW EXPLODE(from_json(pr.products_json, 'array<map<string,string>>')) p AS prod
  WHERE pr.products_json IS NOT NULL
    AND pr.products_json NOT IN ('', '[]', 'null')
    AND p.prod['url_open_access'] IS NOT NULL
    AND COALESCE(p.prod['type'], '') NOT IN ('Octrooi', 'Contract')  -- patents / contracts are not works
),
-- Path 1: salvage a DOI from url_open_access (strip injected whitespace + leading junk),
-- rebuild the canonical https://doi.org/ form, and match openalex_works.doi.
doi_resolved AS (
  SELECT DISTINCT
    pr.project_id,
    w.id AS work_id
  FROM (
    SELECT
      project_id,
      CONCAT('https://doi.org/',
             regexp_extract(lower(regexp_replace(url_oa, '\s+', '')),
                            '(10\.[0-9]{2,}/[^\s"<>]+)', 1)) AS doi_url
    FROM products
    WHERE url_oa RLIKE '(?i)(doi\.org/|10\.[0-9])'
  ) pr
  JOIN openalex.works.openalex_works w
    ON lower(w.doi) = pr.doi_url
  WHERE pr.doi_url <> 'https://doi.org/'
),
-- Path 2: non-DOI URLs -> openalex work locations, accepted only when the URL is unique to
-- exactly one work. Build the unique-URL -> best_doi map, then resolve best_doi -> work.
loc_unique AS (
  SELECT url, MAX(best_doi) AS best_doi
  FROM (
    SELECT regexp_replace(lower(u.url), '/+$', '') AS url, l.best_doi
    FROM openalex.works.locations l
    LATERAL VIEW EXPLODE(l.urls) t AS u
    WHERE u.url IS NOT NULL
      AND l.best_doi IS NOT NULL
      AND NOT lower(u.url) RLIKE 'doi\.org/'
  )
  GROUP BY url
  HAVING COUNT(DISTINCT best_doi) = 1
),
url_resolved AS (
  SELECT DISTINCT
    n.project_id,
    w.id AS work_id
  FROM (
    SELECT
      project_id,
      regexp_replace(lower(regexp_extract(url_oa, '(https?://[^\s]+)', 1)), '/+$', '') AS url
    FROM products
    WHERE url_oa RLIKE '(?i)https?://'
      AND NOT url_oa RLIKE '(?i)doi\.org/'
  ) n
  JOIN loc_unique lu ON lu.url = n.url AND n.url <> ''
  JOIN openalex.works.openalex_works w
    ON lower(w.doi) = CONCAT('https://doi.org/', lower(lu.best_doi))
),
resolved AS (
  SELECT project_id, work_id FROM doi_resolved
  UNION
  SELECT project_id, work_id FROM url_resolved
),
-- Confirm the award entity exists (NWO awards already ingested, provenance 'nwopen') and
-- pick up its funder_id. nwo_awards.funder_award_id == nwo project id (1:1).
with_award AS (
  SELECT
    r.work_id,
    a.funder_id,
    a.funder_award_id
  FROM resolved r
  JOIN openalex.awards.nwo_awards a
    ON a.funder_award_id = r.project_id
  WHERE r.work_id IS NOT NULL
)
SELECT
  work_id,
  funder_id,
  ARRAY_DISTINCT(COLLECT_LIST(funder_award_id)) AS award_ids
FROM with_award
GROUP BY work_id, funder_id

### Sanity checks

In [ ]:
%sql
-- Row counts, key uniqueness, and award fill
SELECT
  COUNT(*)                                              AS total_rows,
  COUNT(DISTINCT CONCAT(work_id, ':', funder_id))      AS distinct_keys,
  COUNT(DISTINCT work_id)                              AS distinct_works,
  COUNT(DISTINCT funder_id)                            AS distinct_funders,
  SUM(SIZE(award_ids))                                 AS total_edges,
  COUNT(CASE WHEN SIZE(award_ids) > 1 THEN 1 END)      AS works_multi_award
FROM openalex.awards.nwo_work_funders

In [ ]:
%sql
-- Composite-key uniqueness (expect 0) and project-level coverage vs. the source population
WITH dup AS (
  SELECT work_id, funder_id, COUNT(*) AS n
  FROM openalex.awards.nwo_work_funders
  GROUP BY work_id, funder_id HAVING n > 1
),
projects_with_products AS (
  SELECT COUNT(*) AS n
  FROM openalex.awards.nwo_projects_raw
  WHERE products_json IS NOT NULL AND products_json NOT IN ('', '[]', 'null')
),
resolved_projects AS (
  SELECT COUNT(DISTINCT a.funder_award_id) AS n
  FROM (
    SELECT EXPLODE(award_ids) AS award_id
    FROM openalex.awards.nwo_work_funders
  ) t
  JOIN openalex.awards.nwo_awards a ON a.funder_award_id = t.award_id
)
SELECT
  (SELECT COUNT(*) FROM dup)                                                AS duplicate_keys,
  (SELECT n FROM projects_with_products)                                    AS projects_with_products,
  (SELECT n FROM resolved_projects)                                         AS projects_with_a_linked_work,
  ROUND(100.0 * (SELECT n FROM resolved_projects)
              / (SELECT n FROM projects_with_products), 1)                  AS pct_projects_covered